In [1]:
import requests
import pandas as pd
import numpy as np
import ccxt
from datetime import datetime, timezone, timedelta
import os

In [2]:
CR_STRATEGY = 'mm3-futurestrading-ha-j'
CR_TOKEN = os.getenv('CR_API_TOKEN_G20JPY')
LOCATION_IDS = {
    12097: 'binance',
    11639: 'bybit'
}

In [3]:
class CoinRoutes:
    def __init__(self, api_token):
        self.session = requests.Session()
        self.session.headers.update({'Authorization': f'Token {api_token}'})
        self.base_url = 'https://g20jpy.coinroutes.io'
    
    def __del__(self):
        self.session.close()

    def _request(self, method, url, **kwargs):
        try:
            response = self.session.request(method, url, **kwargs)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            return None

    def get_positions(self, strategy=None):
        endpoint = '/api/positions/'
        params = {}
        if strategy:
            params['strategy'] = strategy
        result = self._request('get', self.base_url + endpoint, params=params)
        return pd.DataFrame(result)


class TradeService:
    def __init__(self):
        self.session = requests.Session()
        self.base_url = 'http://trade-streaming.ext.g20api.com:8767'
    
    def __del__(self):
        self.session.close()

    def _request(self, method, url, **kwargs):
        try:
            response = self.session.request(method, url, **kwargs)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f'Error: {e}')
            return None
        
    def _get_trades(self, **params):
        endpoint = '/trades/'
        response =  self._request('get', self.base_url + endpoint, params=params)
        return response
        
    def get_trades(self, location_ids, start_dt, end_dt):
        trades = []
        min_clock_id = None
        while True:
            response = self._get_trades(
                location_id = location_ids,
                min_exch_ts = int(start_dt.timestamp() * 1e6),
                max_exch_ts = int(end_dt.timestamp() * 1e6),
                max_clock_id = min_clock_id,
                order_by = 'clock_id'
            )
            if not response:
                break
            trades += response
            min_clock_id = trades[-1]['clock_id']
        trades = pd.DataFrame(trades)
        if trades.empty:
            return trades
        trades['exch_ts'] = pd.to_datetime(trades['exch_ts'], unit="us")
        trades['recv_ts'] = pd.to_datetime(trades['exch_ts'], unit="us")
        trades[['notional','qty','price','fee']] = trades[['notional','qty','price','fee']].astype(float)
        return trades
    

class MarketDataClient:
    def __init__(self):
        self.ccxt_connectors = {}

    def _add_connector(self, exchange: str, timeout: int = 60000): #1min timeout
        self.ccxt_connectors[exchange] = getattr(ccxt, exchange)({'enableRateLimit': True, 'timeout': timeout})

    def get_klines(self, symbol: str, exchange: str, timeframe: str, start_dt: datetime, end_dt: datetime, request_limit=100) -> list:
        if exchange not in self.ccxt_connectors:
            self._add_connector(exchange)
        since = int(start_dt.timestamp()) * 1000
        all_klines = []
        while True:
            klines = self.ccxt_connectors[exchange].fetch_ohlcv(symbol=symbol, timeframe=timeframe, since=since, limit=request_limit)
            if not klines:
                break
            all_klines += klines
            last_ts = datetime.fromtimestamp(klines[-1][0]/1000)
            # if (len(klines) < request_limit) or (last_ts + pd.Timedelta(timeframe) > end_dt):
            if last_ts + pd.Timedelta(timeframe) > end_dt:
                break
            since = klines[-1][0] + 1
        while all_klines:
            last_ts = datetime.fromtimestamp(all_klines[-1][0]/1000)
            if last_ts >= end_dt:
                all_klines.pop()
            else:
                break
        return all_klines
    
    def get_klines_df(self, symbol: str, exchange: str, timeframe: str, start_dt: datetime, end_dt: datetime, request_limit: int = 100) -> pd.DataFrame:
        klines = self.get_klines(symbol, exchange, timeframe, start_dt, end_dt, request_limit)
        df = pd.DataFrame(klines, columns=['timestamp','open','high','low','close','volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        return df

In [4]:
cr = CoinRoutes(api_token=CR_TOKEN)
trade_service = TradeService()
md_client = MarketDataClient()

start_dt = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
end_dt = start_dt + timedelta(days=1)

perp_positions = cr.get_positions(strategy=CR_STRATEGY)

binance_symbols = perp_positions[perp_positions['exchange']=='binancefutures']['quantity_currency'].unique()
bybit_symbols = perp_positions[perp_positions['exchange']=='bybit']['quantity_currency'].unique()

perp_positions['quantity'] = perp_positions['quantity'].astype(float)
perp_positions = perp_positions.rename(columns={'quantity_currency':'symbol', 'quantity':'position'})
perp_positions = perp_positions.groupby('symbol', as_index=False)['position'].sum()

trades = trade_service.get_trades(
    location_ids=LOCATION_IDS.keys(),
    start_dt=start_dt,
    end_dt=end_dt
)

trades['symbol'] = trades['native_inst'].str.extract(r'^P\.([^-]+)-')

binance_loc_ids = [loc_id for loc_id in LOCATION_IDS if LOCATION_IDS[loc_id]=='binance']
binance_symbols = np.union1d(
    binance_symbols,
    trades[trades['location_id'].isin(binance_loc_ids)]['symbol'].unique()
)

bybit_loc_ids = [loc_id for loc_id in LOCATION_IDS if LOCATION_IDS[loc_id]=='bybit']
bybit_symbols = np.union1d(
    bybit_symbols,
    trades[trades['location_id'].isin(bybit_loc_ids)]['symbol'].unique()
)

trades['buy_qty'] = np.where(trades['side']=='B', trades['qty'], 0)
trades['sell_qty'] = np.where(trades['side']=='S', trades['qty'], 0)
trades['buy_notional'] = np.where(trades['side']=='B', trades['notional'], 0)
trades['sell_notional'] = np.where(trades['side']=='S', trades['notional'], 0)

trades = trades.groupby('symbol', as_index=False)[['buy_qty','sell_qty','buy_notional','sell_notional']].sum()

trades['avg_buy_price'] = np.where(trades['buy_qty']==0, 0, trades['buy_notional']  / trades['buy_qty'])
trades['avg_sell_price'] = np.where(trades['sell_qty']==0, 0, trades['sell_notional']  / trades['sell_qty'])

perp_positions = perp_positions.merge(trades, left_on='symbol', right_on='symbol', how='outer').fillna(0)
perp_positions['position(t-1)'] = perp_positions['position'] - perp_positions['buy_qty'] + perp_positions['sell_qty']

perp_positions['price(t-1)'] = np.nan
perp_positions['price'] = np.nan
for idx, row in perp_positions.iterrows():
    if row['symbol'] in binance_symbols:
        md_exch = 'binance'
    elif row['symbol'] in bybit_symbols:
        md_exch = 'bybit'
    else:
        print(f"Exchange unknown for symbol {row['symbol']}. Skipping.")
        continue
    data = md_client.get_klines_df(f'{row['symbol']}/USDT:USDT', md_exch, '1d', start_dt.replace(tzinfo=None), end_dt.replace(tzinfo=None)).set_index('timestamp')
    perp_positions.loc[idx,'price(t-1)'] = data.loc[pd.Timestamp(start_dt.replace(tzinfo=None)), 'open']
    perp_positions.loc[idx,'price'] = data.loc[pd.Timestamp(start_dt.replace(tzinfo=None)), 'close']

perp_positions['positional_pnl'] = (perp_positions['price'] - perp_positions['price(t-1)']) * perp_positions['position(t-1)']
perp_positions['trading_pnl'] = (perp_positions['price'] - perp_positions['avg_buy_price']) * perp_positions['buy_qty'] + (perp_positions['avg_sell_price'] - perp_positions['price']) * perp_positions['sell_qty']
perp_positions['total_pnl'] = perp_positions['positional_pnl'] + perp_positions['trading_pnl']

total_row = {
    'symbol': 'TOTAL',
    'position(t-1)': '',
    'price(t-1)': '',
    'position': '',
    'price': '',
    'buy_qty': '',
    'avg_buy_price': '',
    'sell_qty': '',
    'avg_sell_price': '',
    'positional_pnl': perp_positions['positional_pnl'].sum(),
    'trading_pnl': perp_positions['trading_pnl'].sum(),
    'total_pnl': perp_positions['total_pnl'].sum()
}
perp_positions = pd.concat([perp_positions[total_row.keys()], pd.DataFrame([total_row])], ignore_index=True)

perp_positions.style.format({
    "positional_pnl": "{:,.0f}",
    "trading_pnl": "{:,.0f}",
    "total_pnl": "{:,.0f}",
})

/tmp/ipykernel_2970166/1550747676.py:92: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  if last_ts + pd.Timedelta(timeframe) > end_dt:
/tmp/ipykernel_2970166/1550747676.py:92: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  if last_ts + pd.Timedelta(timeframe) > end_dt:
/tmp/ipykernel_2970166/1550747676.py:92: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  if last_ts + pd.Timedelta(timeframe) > end_dt:
/tmp/ipykernel_2970166/1550747676.py:92: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  if last_ts + pd.Timedelta(timeframe) > end_dt:
/tmp/ipykernel_2970166/1550747676.py:92: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  if last_ts + pd.Timedelta(timeframe) > end_dt:
/tmp/ipykernel_29701

,symbol,position(t-1),price(t-1),position,price,buy_qty,avg_buy_price,sell_qty,avg_sell_price,positional_pnl,trading_pnl,total_pnl
0,AMD,0.000000,550.110000,28.950000,517.500000,28.950000,518.007150,0.000000,0.000000,-0,-15,-15
1,AVAX,-7003.200000,6.710000,-7003.200000,6.710000,0.000000,0.000000,0.000000,0.000000,-0,0,0
2,BABA,0.000000,112.680000,-126.420000,118.660000,0.000000,0.000000,126.420000,118.643713,0,-2,-2
3,BILL,881067.000000,0.041450,1542587.000000,0.034730,1979822.000000,0.036838,1318302.000000,0.037705,"-5,921",-252,"-6,173"
4,BTC,-1.785000,65014.600000,-0.248000,65422.600000,1.537000,65047.602082,0.000000,0.000000,-728,576,-152
5,BTW,-789585.000000,0.058400,-587877.000000,0.060350,258001.000000,0.061440,56293.000000,0.058268,"-1,540",-398,"-1,938"
6,CC,-16302.000000,0.138460,-190055.000000,0.137940,178201.000000,0.140262,351954.000000,0.142597,8,"1,225","1,234"
7,CRCL,-0.000000,63.210000,-154.270000,63.960000,0.000000,0.000000,154.270000,64.820000,-0,133,133
8,ETH,-34.616000,1891.120000,-34.370000,1933.010000,26.203000,1907.297418,25.957000,1926.586824,"-1,450",507,-943
9,GOOGL,0.000000,359.240000,-40.530000,371.900000,0.000000,0.000000,40.530000,370.064722,0,-74,-74
